# loraops on Colab

Trains one LoRA adapter per tenant on a free T4 GPU, evaluates each against the base model, gates it, and stores it in the registry.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Set `REPO_URL` to your GitHub repo (or upload the project zip and unzip it instead).

In [ ]:
REPO_URL = "https://github.com/<you>/<repo>.git"  # <- change me
!git clone -q $REPO_URL project
%cd project
!pip install -q -e ".[train]"

## 2. Check the data

In [ ]:
!loraops validate --tenant acme
!loraops validate --tenant globex

## 3. Run the pipeline for every tenant
`data → train → evaluate (vs base zero-shot, base few-shot, production) → gate → register → promote`

In [ ]:
!loraops run --all

In [ ]:
!loraops registry

## 4. Inspect a model card

In [ ]:
import json
card = json.load(open("registry/acme/v1/card.json"))
print(json.dumps({k: card[k] for k in ("status", "baselines", "gate", "train")}, indent=2)[:3000])

## 5. Try the adapters
One base model in memory, adapters swapped per request: the same idea vLLM uses in production.

In [ ]:
from loraops.config import load_pipeline_config, load_tenant
from loraops.predictors import HFPredictor
from loraops.prompts import build_messages, parse_label
from loraops.registry import Registry, adapter_name

cfg = load_pipeline_config("configs/pipeline.toml")
reg = Registry(cfg.registry.root)
pred = HFPredictor(cfg)

tickets = {
    "acme": ["My order never arrived and tracking is stuck", "Can I pay with a different card?"],
    "globex": ["Our Slack integration stopped posting alerts", "Everything returns 503 since 10 minutes"],
}
for name, texts in tickets.items():
    tenant = load_tenant(name, cfg)
    prod = reg.production(name)
    if not prod:
        print(name, "has no production adapter"); continue
    a = adapter_name(name, prod[0])
    pred.add_adapter(a, reg.adapter_path(name, prod[0]))
    outs = pred.predict([build_messages(tenant, t) for t in texts], adapter=a)
    for t, o in zip(texts, outs):
        print(f"[{a}] {t!r} -> {parse_label(o, tenant.labels)}")

## 6. Retrain and see the gate in action
Add tickets to `data/tenants/acme/tickets.jsonl` and run again: the new version is compared to the one in production on the same frozen test set, and only promoted if it is not worse.

In [ ]:
!loraops run --tenant acme
!loraops registry

## 7. Download the registry
Copy it next to `docker-compose.yml` on a GPU machine to serve all adapters with vLLM.

In [ ]:
!zip -qr registry.zip registry
from google.colab import files
files.download("registry.zip")